In [ ]:
import time
from datetime import date

import pandas as pd
import requests


In [ ]:
acs_api_key = dbutils.secrets.get("pats_token", "default", "acs_api_key")


In [ ]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
# ACS dataset: 5-year estimates. Chosen over acs1 because we want *all*
# counties / CBSAs. acs1 is only published for geographies with >=65k
# population, which drops ~2/3 of US counties. acs5 covers every geography
# at the cost of a rolling 5-year average rather than a single-year snapshot.
DATASET = "acs/acs5"

# Census Data API base. The vintage year and dataset are PATH segments, not
# query params, so the request URL is built as:
#   https://api.census.gov/data/{year}/{DATASET}?get=...&for=...&in=...&key=...
API_BASE = "https://api.census.gov/data"

# First year to pull. Latest year is auto-discovered by walking forward until a
# vintage is not published yet (see fetch_acs_range).
START_YEAR = 2018

# Variables to request, confirmed against the current Census variable metadata
# (https://api.census.gov/data/2024/acs/acs5/variables/<code>.json):
#   B25064_001E -> "Estimate!!Median gross rent"  (Median Gross Rent, dollars)
#   B19013_001E -> "Estimate!!Median household income in the past 12 months
#                   (in <year> inflation-adjusted dollars)"
# The API accepts multiple comma-separated variables in a single get= call,
# so both come back in ONE request per year (plus NAME for a readable label).
VARIABLES = {
    "B25064_001E": "median_gross_rent",
    "B19013_001E": "median_household_income",
}

# Geography is parameterized as a for= / in= pair so it can be swapped without
# touching the request logic. Default: every county in every state.
# NOTE: BRONZE_TABLE below is county-grain -- switching to CBSA needs its own
# target table (the write cell asserts on this).
GEOGRAPHY_COUNTY = {
    "for": "county:*",
    "in": "state:*",
}
GEOGRAPHY_CBSA = {
    "for": "metropolitan statistical area/micropolitan statistical area:*",
    "in": None,
}

GEOGRAPHY = GEOGRAPHY_COUNTY

# ---------------------------------------------------------------------------
# Bronze target
# ---------------------------------------------------------------------------
# Medallion layer = Bronze: land the data with source fidelity. Values are kept
# as the raw strings the API returns. Numeric casting and Census "jam" value
# handling (-666666666 etc. -> NULL) are deliberately deferred to Silver.
BRONZE_TABLE = "pats_token.bronze.acs_housing_income_county_bronze"
SOURCE_TAG = "census_acs_api"

# Full-refresh write: every run re-pulls the whole START_YEAR..latest panel and
# overwrites the table. ACS revises prior vintages, so a full replace keeps us
# consistent with the source. Data volume is tiny (~7 yrs x ~3,220 counties).

# Pre-write sanity gate: minimum rows expected per year (~3,220 US counties).
MIN_ROWS_PER_YEAR = 3000

# Transient-failure retry policy for the API calls.
MAX_RETRIES = 3
RETRY_BACKOFF_SECS = 2  # sleep = RETRY_BACKOFF_SECS ** attempt


In [ ]:
# ---------------------------------------------------------------------------
# Request logic
# ---------------------------------------------------------------------------
NOT_RELEASED_STATUS = {404}  # vintage doesn't exist yet -> stop the walk, don't error


def fetch_acs_year(year, variables, geography, api_key, dataset=DATASET, timeout=30):
    """Pull all requested variables for one vintage year in a single request.

    Returns the raw Census response (list of lists, row 0 = header) on success,
    or None if that vintage is not published yet (HTTP 404).

    Transient failures (network errors, 5xx, 429, rate limits, and the Census
    "200 with an HTML error body" case) are retried MAX_RETRIES times with
    exponential backoff. A year that still fails raises RuntimeError rather than
    being silently skipped.
    """
    url = f"{API_BASE}/{year}/{dataset}"
    params = {
        "get": ",".join(["NAME", *variables]),
        "for": geography["for"],
        "key": api_key,
    }
    if geography.get("in"):
        params["in"] = geography["in"]

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, params=params, timeout=timeout)
            if resp.status_code in NOT_RELEASED_STATUS:
                return None
            resp.raise_for_status()
            data = resp.json()  # 200-with-HTML-error body raises ValueError here
            if not isinstance(data, list) or len(data) < 2:
                raise ValueError(f"empty or non-tabular body: {resp.text[:200]!r}")
            return data
        except (requests.RequestException, ValueError) as exc:
            if attempt == MAX_RETRIES:
                raise RuntimeError(
                    f"ACS {dataset} {year} failed after {MAX_RETRIES} attempts: {exc}"
                ) from exc
            time.sleep(RETRY_BACKOFF_SECS ** attempt)


def fetch_acs_range(variables, geography, api_key, start_year=START_YEAR, dataset=DATASET):
    """Pull start_year .. most-recent-published vintage, one request per year.

    Walks forward from start_year and stops only on a genuine "not released yet"
    (404) response. Any other failure propagates out of fetch_acs_year, so the
    range is never silently truncated by a transient error. ACS 5-year for year
    Y is released ~Dec of Y+1 (2020-2024 landed 2026-01-29), so next calendar
    year is the ceiling.
    """
    raw_by_year = {}
    for year in range(start_year, date.today().year + 1):
        data = fetch_acs_year(year, variables, geography, api_key, dataset=dataset)
        if data is None:
            break
        raw_by_year[year] = data
        time.sleep(0.2)  # be polite to the API
    if not raw_by_year:
        raise RuntimeError(f"No ACS {dataset} vintage available at or after {start_year}")
    return raw_by_year


raw_by_year = fetch_acs_range(list(VARIABLES), GEOGRAPHY, acs_api_key)
pulled_years = sorted(raw_by_year)
print(f"Pulled {DATASET} for {pulled_years[0]}-{pulled_years[-1]} "
      f"({len(pulled_years)} requests)")


In [ ]:
# ---------------------------------------------------------------------------
# Reshape raw JSON -> tidy frame: one row per (geography, year)
# ---------------------------------------------------------------------------
# Bronze = source fidelity: the estimate columns stay as the raw strings the
# API returned (including Census jam values like "-666666666"). Casting those
# to numbers and nulling the jam values is a Silver-layer job.
def parse_acs_year(rows, year, variables=VARIABLES):
    """rows[0] is the header, rows[1:] the records. The geography id columns
    (state + county for the default query, or the CBSA code) are whatever
    trails the requested variables in the response header."""
    header = rows[0]
    df = pd.DataFrame(rows[1:], columns=header)

    geo_id_cols = [c for c in header if c not in ({"NAME"} | set(variables))]
    df["geo_id"] = df[geo_id_cols].agg("".join, axis=1)  # e.g. state+county FIPS
    df["year"] = year
    df = df.rename(columns={"NAME": "geography", **variables})

    ordered = ["geo_id", "geography", *geo_id_cols, "year", *variables.values()]
    return df[ordered]


acs_df = pd.concat(
    [parse_acs_year(rows, year) for year, rows in sorted(raw_by_year.items())],
    ignore_index=True,
)
acs_df = acs_df.sort_values(["geo_id", "year"]).reset_index(drop=True)
acs_df


In [ ]:
# ---------------------------------------------------------------------------
# Pre-write sanity gates -- fail loudly rather than promote a bad table
# ---------------------------------------------------------------------------
rows_per_year = acs_df.groupby("year").size()
print(rows_per_year)

# 1. Every year returned a full geography set (guards against a 200 response
#    that silently contains only part of the data).
assert (rows_per_year >= MIN_ROWS_PER_YEAR).all(), (
    f"row count below {MIN_ROWS_PER_YEAR} for some year:\n{rows_per_year}"
)

# 2. One row per (geography, year).
dupes = acs_df.duplicated(["geo_id", "year"]).sum()
assert dupes == 0, f"{dupes} duplicate (geo_id, year) rows"

# 3. No gap in the year range (walk-forward stopped cleanly, not mid-sequence).
years_seen = sorted(rows_per_year.index)
assert years_seen == list(range(years_seen[0], years_seen[-1] + 1)), (
    f"gap in year coverage: {years_seen}"
)

# 4. Schema is exactly what the Bronze table expects.
EXPECTED_COLUMNS = [
    "geo_id", "geography", "state", "county", "year",
    "median_gross_rent", "median_household_income",
]
assert list(acs_df.columns) == EXPECTED_COLUMNS, (
    f"unexpected columns: {list(acs_df.columns)}"
)

print(f"\nOK: {len(acs_df)} rows, years {years_seen[0]}-{years_seen[-1]}")


In [ ]:
# ---------------------------------------------------------------------------
# Write to the Bronze Unity Catalog table (full overwrite each run)
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# This notebook + table are county-grain. A CBSA pull has a different geo id
# column and belongs in its own Bronze table.
assert GEOGRAPHY == GEOGRAPHY_COUNTY, (
    f"{BRONZE_TABLE} is county-grain; use a different table for other geographies"
)

# Explicit schema so the table's shape never drifts with pandas dtype inference.
# Estimate columns are StringType on purpose (Bronze keeps source strings).
bronze_schema = StructType([
    StructField("geo_id", StringType(), nullable=False),
    StructField("geography", StringType(), nullable=True),
    StructField("state", StringType(), nullable=True),
    StructField("county", StringType(), nullable=True),
    StructField("year", IntegerType(), nullable=False),
    StructField("median_gross_rent", StringType(), nullable=True),
    StructField("median_household_income", StringType(), nullable=True),
])

pdf = acs_df.copy()
pdf["year"] = pdf["year"].astype("int32")

sdf = (
    spark.createDataFrame(pdf, schema=bronze_schema)
    .withColumn("_ingested_at", F.current_timestamp())   # one value per run
    .withColumn("_source", F.lit(SOURCE_TAG))
    .withColumn("_source_dataset", F.lit(DATASET))       # acs5 vs acs1 distinguishable
    .withColumn("_source_url", F.lit(API_BASE))
)

(
    sdf.write
    .format("delta")
    .mode("overwrite")
    # Keep schema evolution OFF: a dtype/column regression should fail the write
    # loudly. Flip to "true" for ONE run only when intentionally changing schema.
    .option("overwriteSchema", "false")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Wrote {sdf.count()} rows to {BRONZE_TABLE}")
display(spark.table(BRONZE_TABLE).orderBy("geo_id", "year"))
